In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.text_normalize import normalize_text

In [2]:
def check(raw, expected):
    got = normalize_text(raw)
    assert got == expected, f"expected {expected!r}, got {got!r}"

check("\u0643\u0627\u0644\u0627", "\u06A9\u0627\u0644\u0627")      # Arabic kaf -> Persian kaf
check("خوب   بود\n\nعالی", "خوب بود عالی")                           # spaces and line breaks
check("می\u200cخواهم", "می\u200cخواهم")                              # half-space must stay
check("a\u200bb\u200fc", "abc")                                      # invisible characters removed
check("قیمت ۱۲۰۰ و ٣٤ و 56", "قیمت 1200 و 34 و 56")                # digits unified
check("سایت https://x.com/a?b=1 خوب", "سایت خوب")                   # URL removed
check("عالی 😍", "عالی 😍")                                          # emoji kept
check("نه، اصلاً خوب نیست", "نه، اصلاً خوب نیست")                  # negation kept
check("  \t خوبي\u00a0و\r\nكيفيت ", "خوبی و کیفیت")                 # mixed case
check(None, "")
print("all checks passed")

all checks passed


In [3]:
import duckdb

con = duckdb.connect()
con.sql("""
    CREATE OR REPLACE TEMP VIEW comments AS
    SELECT * FROM read_csv('../data/digikala-comments.csv', all_varchar=true)
""")

sample = con.sql(r"""
    SELECT id, body
    FROM comments
    WHERE body LIKE '%ي%' OR body LIKE '%ك%'
       OR regexp_matches(body, '[\x{06F0}-\x{06F9}]')
    LIMIT 8
""").df()

sample["normalized"] = sample["body"].apply(normalize_text)
sample

,id,body,normalized
0,45354820,به نظرم چیز خوب و خفنیه مخصوصا با قیمت ۱۰ ت \r...,به نظرم چیز خوب و خفنیه مخصوصا با قیمت 10 ت من...
1,17911442,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...,من چند رنگش رو سفارش دادم.خیلی خوش رنگ بودن تو...
2,47241861,اینا رو بیرون دونه ای ۵ یا هفت میفروشن \r\nالا...,اینا رو بیرون دونه ای 5 یا هفت میفروشن الان 6 ...
3,25986578,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...,خیلی خوشگل و کاربردی هستش و اندازه مناسبی داره...
4,29582488,سلام من سایه آکوآ رنگ ۷۴ سفارش دادم طبق نظرلت ...,سلام من سایه آکوآ رنگ 74 سفارش دادم طبق نظرلت ...
5,19945372,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...,خیلی نسبت به قیمتش فقط دوستان بدونین این ماسک ...
6,53894544,خوبه\r\nهمشون سالم بودن\r\nتعداد درست ارسال شد...,خوبه همشون سالم بودن تعداد درست ارسال شده بود ...
7,52130458,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن\nق...,تعداد توپک ها زیاده و هر کدوم حجم خوبی دارن قی...


In [4]:
from duckdb.sqltypes import VARCHAR

con.create_function("normalize_text", normalize_text, [VARCHAR], VARCHAR)

con.sql("SELECT normalize_text('خوبي   و كيفيت ۱۲۳') AS result").show()

┌──────────────────┐
│      result      │
│     varchar      │
├──────────────────┤
│ خوبی و کیفیت 123 │
└──────────────────┘



In [5]:
from pathlib import Path

INTERIM = Path("..") / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)
NORMALIZED = INTERIM / "comments_normalized.parquet"

con.sql(f"""
    COPY (
        SELECT id, product_id, rate, recommendation_status, is_buyer,
               body, normalize_text(body) AS body_norm
        FROM comments
    ) TO '{NORMALIZED.as_posix()}' (FORMAT PARQUET)
""")

CountBIGINT
6156289


In [6]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE body <> body_norm) AS changed_rows,
           COUNT(*) FILTER (WHERE body_norm LIKE '%ي%' OR body_norm LIKE '%ك%') AS arabic_letters_left,
           COUNT(*) FILTER (WHERE body_norm = '') AS empty_after
    FROM read_parquet('{NORMALIZED.as_posix()}')
""").show()

┌─────────┬──────────────┬─────────────────────┬─────────────┐
│ n_rows  │ changed_rows │ arabic_letters_left │ empty_after │
│  int64  │    int64     │        int64        │    int64    │
├─────────┼──────────────┼─────────────────────┼─────────────┤
│ 6156289 │      1624138 │                   0 │           0 │
└─────────┴──────────────┴─────────────────────┴─────────────┘



In [7]:
con.sql(f"""
    WITH b AS (
        SELECT body_norm, COUNT(*) AS n
        FROM read_parquet('{NORMALIZED.as_posix()}')
        GROUP BY body_norm
    )
    SELECT COUNT(*) AS distinct_bodies,
           COUNT(*) FILTER (WHERE n > 1) AS bodies_repeated,
           SUM(n) FILTER (WHERE n > 1)   AS rows_in_repeated_bodies
    FROM b
""").show()

┌─────────────────┬─────────────────┬─────────────────────────┐
│ distinct_bodies │ bodies_repeated │ rows_in_repeated_bodies │
│      int64      │      int64      │         int128          │
├─────────────────┼─────────────────┼─────────────────────────┤
│         4027268 │          193458 │                 2322479 │
└─────────────────┴─────────────────┴─────────────────────────┘



In [8]:
con.sql(f"""
    WITH t AS (
        SELECT body_norm,
               COUNT(DISTINCT recommendation_status) AS n_labels,
               COUNT(*) AS n
        FROM read_parquet('{NORMALIZED.as_posix()}')
        WHERE recommendation_status <> 'nan'
        GROUP BY body_norm
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS repeated_labelled_bodies,
           COUNT(*) FILTER (WHERE n_labels > 1) AS with_conflicting_labels
    FROM t
""").show()

┌──────────────────────────┬─────────────────────────┐
│ repeated_labelled_bodies │ with_conflicting_labels │
│          int64           │          int64          │
├──────────────────────────┼─────────────────────────┤
│                   169985 │                   31393 │
└──────────────────────────┴─────────────────────────┘



## Normalization rules (src/text_normalize.py)

Each rule follows a prevalence count from notebook 01:
- Arabic yeh/kaf -> Persian yeh/kaf (48,719 and 25,363 rows).
- Persian and Arabic-Indic digits -> Latin digits (provisional; to be re-checked against the model tokenizer).
- Invisible characters other than the half-space (U+200C) are removed (1,005 rows). The half-space is kept (193,672 rows).
- URLs are removed (9 rows).
- Whitespace runs, tabs and line breaks become one space (210,707 and 567,096 rows).
- Emojis, punctuation, negation words and stop words are kept unchanged.

In [9]:
con.sql(rf"""
    SELECT
        COUNT(*) FILTER (WHERE body <> body_norm) AS changed_rows,
        COUNT(*) FILTER (WHERE body <> body_norm
            AND trim(regexp_replace(body, '[\s\x{{00A0}}]+', ' ', 'g')) = body_norm) AS whitespace_only,
        COUNT(*) FILTER (WHERE regexp_matches(body, '^\s|\s$')) AS edge_whitespace,
        COUNT(*) FILTER (WHERE body LIKE '%' || chr(160) || '%') AS no_break_space
    FROM read_parquet('{NORMALIZED.as_posix()}')
""").show()

┌──────────────┬─────────────────┬─────────────────┬────────────────┐
│ changed_rows │ whitespace_only │ edge_whitespace │ no_break_space │
│    int64     │      int64      │      int64      │     int64      │
├──────────────┼─────────────────┼─────────────────┼────────────────┤
│      1624138 │         1378199 │          895072 │            234 │
└──────────────┴─────────────────┴─────────────────┴────────────────┘



## Normalization results

- All 6,156,289 rows were normalized; 1,624,138 rows (26.4%) changed and none became empty.
- No Arabic yeh/kaf remain after normalization (before: 48,719 and 25,363 rows).
- Distinct texts fell from 4,136,258 to 4,027,268. Rows whose text occurs more than once rose from 2,217,776 (36.0%) to 2,322,479 (37.7%), while the number of repeated texts fell from 197,745 to 193,458 because variants merged into one text.
- Repeated labelled texts: 169,985, of which 31,393 (18.5%) have conflicting labels (before normalization 31,942 of 173,905, 18.4%).

In [10]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(*) FILTER (WHERE product_id IS NULL OR product_id IN ('nan', '')) AS missing_product,
           COUNT(DISTINCT product_id) AS distinct_products
    FROM read_parquet('{NORMALIZED.as_posix()}')
""").show()

┌─────────┬─────────────────┬───────────────────┐
│ n_rows  │ missing_product │ distinct_products │
│  int64  │      int64      │       int64       │
├─────────┼─────────────────┼───────────────────┤
│ 6156289 │               0 │            331599 │
└─────────┴─────────────────┴───────────────────┘



In [11]:
con.sql(f"""
    SELECT rate_band, recommendation_status, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY rate_band), 1) AS pct_in_band
    FROM (
        SELECT recommendation_status,
               CASE WHEN TRY_CAST(rate AS DOUBLE) BETWEEN 1 AND 5 THEN
                        CASE WHEN TRY_CAST(rate AS DOUBLE) < 2.5 THEN '1 low (1 to <2.5)'
                             WHEN TRY_CAST(rate AS DOUBLE) < 4   THEN '2 mid (2.5 to <4)'
                             ELSE '3 high (4 to 5)' END
                    ELSE '0 zero or invalid' END AS rate_band
        FROM read_parquet('{NORMALIZED.as_posix()}')
        WHERE recommendation_status <> 'nan'
    )
    GROUP BY rate_band, recommendation_status
    ORDER BY rate_band, n DESC
""").show(max_rows=40)

┌───────────────────┬───────────────────────┬─────────┬─────────────┐
│     rate_band     │ recommendation_status │    n    │ pct_in_band │
│      varchar      │        varchar        │  int64  │   double    │
├───────────────────┼───────────────────────┼─────────┼─────────────┤
│ 0 zero or invalid │ recommended           │  426648 │        84.0 │
│ 0 zero or invalid │ not_recommended       │   60375 │        11.9 │
│ 0 zero or invalid │ no_idea               │   20978 │         4.1 │
│ 1 low (1 to <2.5) │ not_recommended       │  364023 │        75.3 │
│ 1 low (1 to <2.5) │ no_idea               │   83237 │        17.2 │
│ 1 low (1 to <2.5) │ recommended           │   36355 │         7.5 │
│ 2 mid (2.5 to <4) │ recommended           │  491915 │        51.9 │
│ 2 mid (2.5 to <4) │ no_idea               │  388062 │        40.9 │
│ 2 mid (2.5 to <4) │ not_recommended       │   67870 │         7.2 │
│ 3 high (4 to 5)   │ recommended           │ 3207921 │        96.6 │
│ 3 high (4 to 5)   

In [12]:
con.sql(f"""
    WITH top_texts AS (
        SELECT body_norm
        FROM read_parquet('{NORMALIZED.as_posix()}')
        WHERE recommendation_status <> 'nan'
        GROUP BY body_norm
        ORDER BY COUNT(*) DESC
        LIMIT 12
    )
    SELECT body_norm, recommendation_status, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY body_norm), 1) AS pct_of_text
    FROM read_parquet('{NORMALIZED.as_posix()}')
    WHERE recommendation_status <> 'nan'
      AND body_norm IN (SELECT body_norm FROM top_texts)
    GROUP BY body_norm, recommendation_status
    ORDER BY body_norm, n DESC
""").show(max_rows=60)

┌────────────────────┬───────────────────────┬───────┬─────────────┐
│     body_norm      │ recommendation_status │   n   │ pct_of_text │
│      varchar       │        varchar        │ int64 │   double    │
├────────────────────┼───────────────────────┼───────┼─────────────┤
│ خوب                │ recommended           │ 45602 │        92.8 │
│ خوب                │ no_idea               │  3290 │         6.7 │
│ خوب                │ not_recommended       │   232 │         0.5 │
│ خوب بود            │ recommended           │ 64000 │        93.1 │
│ خوب بود            │ no_idea               │  4439 │         6.5 │
│ خوب بود            │ not_recommended       │   270 │         0.4 │
│ خوبه               │ recommended           │ 57121 │        93.1 │
│ خوبه               │ no_idea               │  4002 │         6.5 │
│ خوبه               │ not_recommended       │   243 │         0.4 │
│ خیلی خوب بود       │ recommended           │ 15357 │        99.0 │
│ خیلی خوب بود       │ no_idea    

In [13]:
con.sql(rf"""
    SELECT
        COUNT(*) FILTER (WHERE NOT regexp_matches(body_norm, '[\x{{0600}}-\x{{06FF}}]')) AS no_persian_letters,
        COUNT(*) FILTER (WHERE regexp_matches(body_norm, '[A-Za-z]')
                           AND NOT regexp_matches(body_norm, '[\x{{0600}}-\x{{06FF}}]')) AS latin_only,
        COUNT(*) FILTER (WHERE regexp_matches(body_norm, 'ی{{3,}}|ا{{3,}}|و{{3,}}|ه{{3,}}|ب{{3,}}|ع{{3,}}|خ{{3,}}|ن{{3,}}')) AS elongated
    FROM read_parquet('{NORMALIZED.as_posix()}')
""").show()

┌────────────────────┬────────────┬───────────┐
│ no_persian_letters │ latin_only │ elongated │
│       int64        │   int64    │   int64   │
├────────────────────┼────────────┼───────────┤
│               4940 │       3944 │    114336 │
└────────────────────┴────────────┴───────────┘



In [14]:
con.sql("SELECT created_at FROM comments LIMIT 5").show()
con.sql("SELECT MIN(created_at) AS first, MAX(created_at) AS last FROM comments").show()

┌────────────────┐
│   created_at   │
│    varchar     │
├────────────────┤
│ 23 شهریور 1402 │
│ 16 تیر 1399    │
│ 26 مرداد 1401  │
│ 28 اسفند 1399  │
│ 12 شهریور 1402 │
└────────────────┘

┌─────────────┬────────────┐
│    first    │    last    │
│   varchar   │  varchar   │
├─────────────┼────────────┤
│ 1 آبان 1395 │ 9 مهر 1402 │
└─────────────┴────────────┘



In [15]:
sample = con.sql(f"""
    SELECT recommendation_status, rate, left(body_norm, 120) AS text
    FROM (
        SELECT *, ROW_NUMBER() OVER (
                   PARTITION BY recommendation_status
                   ORDER BY hash(CAST(id AS VARCHAR) || '2026')) AS rn
        FROM read_parquet('{NORMALIZED.as_posix()}')
        WHERE recommendation_status <> 'nan'
    )
    WHERE rn <= 10
    ORDER BY recommendation_status, rn
""").df()
sample

,recommendation_status,rate,text
0,no_idea,3.75,به کسی کادو دادم. نمیدادم الان در چه وضعیه، ول...
1,no_idea,3.0,خیلی تو دست قشنگه و ظریفه ولی متاسفانه رنگش خی...
2,no_idea,3.0,من برای بعد از پلاسما گرفتم و هنوز استفاده نکر...
3,no_idea,3.0,معمولیه و زیاد جالب نیست
4,no_idea,3.0,بچه ها انتخاب کردند. راضی بودند.
5,no_idea,3.0,قشنگ بود ولی جنس زیاد خوبی نداشت
6,no_idea,2.0,سایز خیلی کوچیک
7,no_idea,3.0,جنسش خوبه ولی باید دو سایز بزرگتر سفارش داد
8,no_idea,0.0,بدنیست
9,no_idea,3.0,سایز لارج گرفتم ولی اندازه مدیوم هستش با اینکه...


In [16]:
con.sql("""
    CREATE OR REPLACE TEMP TABLE conflicting_ids AS
    SELECT id FROM comments
    WHERE id IN (SELECT id FROM comments GROUP BY id HAVING COUNT(*) > 1)
    GROUP BY id
    HAVING COUNT(DISTINCT md5(concat_ws('|', title, body, created_at, rate,
           recommendation_status, product_id, seller_code))) > 1
""")
con.sql("SELECT COUNT(*) AS conflicting_ids FROM conflicting_ids").show()

┌─────────────────┐
│ conflicting_ids │
│      int64      │
├─────────────────┤
│             387 │
└─────────────────┘



In [17]:
CLEAN = INTERIM / "comments_clean.parquet"

con.sql(f"""
    COPY (
        SELECT id, product_id, is_buyer, created_at, body,
               normalize_text(body) AS body_norm,
               CASE WHEN recommendation_status = 'nan' THEN NULL
                    ELSE recommendation_status END AS label,
               TRY_CAST(rate AS DOUBLE) AS rate_num,
               CASE WHEN TRY_CAST(rate AS DOUBLE) BETWEEN 1 AND 5 THEN
                        CASE WHEN TRY_CAST(rate AS DOUBLE) < 2.5 THEN 'low'
                             WHEN TRY_CAST(rate AS DOUBLE) < 4   THEN 'mid'
                             ELSE 'high' END
                    ELSE NULL END AS rate_band
        FROM comments
        WHERE id NOT IN (SELECT id FROM conflicting_ids)
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id) = 1
    ) TO '{CLEAN.as_posix()}' (FORMAT PARQUET)
""")

CountBIGINT
6152673


In [18]:
con.sql(rf"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT id) AS distinct_ids,
           COUNT(*) FILTER (WHERE label IS NOT NULL) AS labelled,
           COUNT(*) FILTER (WHERE rate_band IS NOT NULL) AS with_rate_band,
           COUNT(*) FILTER (WHERE regexp_matches(body_norm, '^\s|\s$')) AS edge_ws_left
    FROM read_parquet('{CLEAN.as_posix()}')
""").show()

┌─────────┬──────────────┬──────────┬────────────────┬──────────────┐
│ n_rows  │ distinct_ids │ labelled │ with_rate_band │ edge_ws_left │
│  int64  │    int64     │  int64   │     int64      │    int64     │
├─────────┼──────────────┼──────────┼────────────────┼──────────────┤
│ 6152673 │      6152673 │  5258628 │        5618513 │            0 │
└─────────┴──────────────┴──────────┴────────────────┴──────────────┘



## Cleaning rules

- Exact duplicate rows (same id and identical title, body, created_at, rate, recommendation_status, product_id and seller_code): one copy kept, the others dropped.
- Ids that occur with different content (387 ids, 774 rows): excluded entirely, because the correct version cannot be determined.
- Recommendation label "nan" is stored as a true missing value; those rows stay in the file but are not used as labelled examples.
- Rating: only values from 1 to 5 are valid (instructor decision). Zero, values between 0 and 1 and values above 5 get no rating band but the row is kept for the recommendation task.
- Rating bands: low [1, 2.5), mid [2.5, 4), high [4, 5].
- Review text: normalized with src/text_normalize.py (see notebook section "Normalization rules").

In [19]:
con.sql("""
    SELECT TRY_CAST(split_part(created_at, ' ', 3) AS INTEGER) AS jalali_year,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) FILTER (WHERE recommendation_status = 'recommended')
                 / NULLIF(COUNT(*) FILTER (WHERE recommendation_status <> 'nan'), 0), 1) AS pct_recommended_of_labelled,
           ROUND(100.0 * COUNT(*) FILTER (WHERE recommendation_status = 'nan') / COUNT(*), 1) AS pct_missing_label
    FROM comments
    GROUP BY jalali_year
    ORDER BY jalali_year
""").show()

┌─────────────┬─────────┬─────────────────────────────┬───────────────────┐
│ jalali_year │    n    │ pct_recommended_of_labelled │ pct_missing_label │
│    int32    │  int64  │           double            │      double       │
├─────────────┼─────────┼─────────────────────────────┼───────────────────┤
│        1395 │    3575 │                        65.0 │              97.1 │
│        1396 │    6357 │                        69.7 │              97.4 │
│        1397 │   16693 │                        61.4 │              48.0 │
│        1398 │  138645 │                        75.7 │              13.4 │
│        1399 │  684321 │                        78.0 │               5.6 │
│        1400 │ 1253484 │                        78.5 │               4.8 │
│        1401 │ 2165894 │                        79.1 │              26.3 │
│        1402 │ 1887320 │                        80.3 │              10.1 │
└─────────────┴─────────┴─────────────────────────────┴───────────────────┘



In [20]:
con.sql(f"""
    WITH t AS (
        SELECT body_norm, label, COUNT(*) AS n
        FROM read_parquet('{CLEAN.as_posix()}')
        WHERE label IS NOT NULL
        GROUP BY body_norm, label
    ),
    m AS (
        SELECT body_norm, SUM(n) AS total, MAX(n) AS best
        FROM t
        GROUP BY body_norm
        HAVING SUM(n) > 1
    )
    SELECT COUNT(*) AS repeated_texts,
           SUM(total) AS rows_covered,
           ROUND(100.0 * SUM(best) / SUM(total), 2) AS majority_label_accuracy_pct
    FROM m
""").show()

┌────────────────┬──────────────┬─────────────────────────────┐
│ repeated_texts │ rows_covered │ majority_label_accuracy_pct │
│     int64      │    int128    │           double            │
├────────────────┼──────────────┼─────────────────────────────┤
│         168092 │      1918136 │                       92.99 │
└────────────────┴──────────────┴─────────────────────────────┘



In [21]:
con.sql(rf"""
    SELECT label, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) FILTER (WHERE regexp_matches(' ' || body_norm || ' ',
                 '[ .،!؟]+(ولی|اما|متاسفانه)[ .،!؟]+')) / COUNT(*), 1) AS pct_with_contrast_word
    FROM read_parquet('{CLEAN.as_posix()}')
    WHERE label IS NOT NULL
    GROUP BY label
    ORDER BY label
""").show()

┌─────────────────┬─────────┬────────────────────────┐
│      label      │    n    │ pct_with_contrast_word │
│     varchar     │  int64  │         double         │
├─────────────────┼─────────┼────────────────────────┤
│ no_idea         │  593682 │                   28.1 │
│ not_recommended │  504482 │                   16.5 │
│ recommended     │ 4160464 │                   10.1 │
└─────────────────┴─────────┴────────────────────────┘



In [22]:
SPLIT_SEED = 2026
PRODUCT_SPLIT = INTERIM / "product_split.parquet"

con.execute(f"""
COPY (
  WITH prod AS (
    SELECT product_id, COUNT(*) AS n_reviews
    FROM read_parquet('{CLEAN.as_posix()}')
    GROUP BY product_id
  ),
  ordered AS (
    SELECT product_id, n_reviews,
           SUM(n_reviews) OVER (
             ORDER BY md5(CAST(product_id AS VARCHAR) || '|{SPLIT_SEED}'), product_id
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) - n_reviews AS rows_before,
           SUM(n_reviews) OVER () AS rows_total
    FROM prod
  )
  SELECT product_id, n_reviews,
         CASE WHEN rows_before < 0.70 * rows_total THEN 'train'
              WHEN rows_before < 0.85 * rows_total THEN 'validation'
              ELSE 'test' END AS split
  FROM ordered
) TO '{PRODUCT_SPLIT.as_posix()}' (FORMAT PARQUET);
""")
print("done")

done


In [23]:
con.execute(f"""
SELECT split,
       COUNT(*) AS n_products,
       SUM(n_reviews) AS n_reviews,
       ROUND(100.0 * SUM(n_reviews) / SUM(SUM(n_reviews)) OVER (), 2) AS pct_reviews
FROM read_parquet('{PRODUCT_SPLIT.as_posix()}')
GROUP BY split
ORDER BY split
""").df()

,split,n_products,n_reviews,pct_reviews
0,test,49515,922896.0,15.0
1,train,231933,4306881.0,70.0
2,validation,50149,922896.0,15.0


In [24]:
con.execute(f"""
SELECT COUNT(*) FROM (
  SELECT c.product_id
  FROM read_parquet('{CLEAN.as_posix()}') c
  JOIN read_parquet('{PRODUCT_SPLIT.as_posix()}') s USING (product_id)
  GROUP BY c.product_id
  HAVING COUNT(DISTINCT s.split) > 1
)
""").fetchall()[0][0]

0

In [25]:
con.execute(f"""
SELECT s.split, COALESCE(c.label, '(missing)') AS label, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY s.split), 2) AS pct_in_split
FROM read_parquet('{CLEAN.as_posix()}') c
JOIN read_parquet('{PRODUCT_SPLIT.as_posix()}') s USING (product_id)
GROUP BY s.split, label
ORDER BY label, s.split
""").df()

,split,label,n,pct_in_split
0,test,(missing),134493,14.57
1,train,(missing),624981,14.51
2,validation,(missing),134571,14.58
3,test,no_idea,88816,9.62
4,train,no_idea,416298,9.67
5,validation,no_idea,88568,9.60
6,test,not_recommended,74409,8.06
7,train,not_recommended,354431,8.23
8,validation,not_recommended,75642,8.20
9,test,recommended,625178,67.74


In [26]:
import hashlib, math
import pandas as pd

SPLIT_SEED = 2026
MASK_SEED = "digikala-semi-supervised-v1"
MASK_RATE = 0.20

def stable_hash(row_id: str) -> int:
    key = f"{MASK_SEED}|{row_id}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:16], 16)

train_labeled = con.execute(f"""
SELECT c.id, c.label
FROM read_parquet('{CLEAN.as_posix()}') c
JOIN read_parquet('{PRODUCT_SPLIT.as_posix()}') s USING (product_id)
WHERE s.split = 'train' AND c.label IS NOT NULL
""").df()

train_labeled["h"] = train_labeled["id"].map(stable_hash)

parts = []
masked_counts = {}
class_counts = {}
for cls, grp in train_labeled.groupby("label"):
    k = math.floor(MASK_RATE * len(grp))
    class_counts[cls] = len(grp)
    masked_counts[cls] = k
    parts.append(grp.sort_values(["h", "id"]).head(k))
masked = pd.concat(parts)

MASKED = INTERIM / "masked_ids.parquet"
masked[["id", "label"]].to_parquet(MASKED)
print(class_counts)
print(masked_counts)

{'no_idea': 416298, 'not_recommended': 354431, 'recommended': 2911171}
{'no_idea': 83259, 'not_recommended': 70886, 'recommended': 582234}


In [27]:
OUT = Path("..") / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

con.execute(f"""
CREATE OR REPLACE TEMP VIEW rows_split AS
SELECT c.id, c.body, c.body_norm,
       c.label AS recommendation_status,
       CASE c.rate_band WHEN 'mid' THEN 'medium' ELSE c.rate_band END AS rating_band,
       c.rate_num AS rate,
       s.split
FROM read_parquet('{CLEAN.as_posix()}') c
JOIN read_parquet('{PRODUCT_SPLIT.as_posix()}') s USING (product_id)
""")
con.execute(f"CREATE OR REPLACE TEMP VIEW masked AS SELECT id FROM read_parquet('{MASKED.as_posix()}')")

FILES = {
 "train_supervised": """
    SELECT id, body, body_norm, recommendation_status, rating_band, rate FROM rows_split
    WHERE split = 'train' AND id NOT IN (SELECT id FROM masked)
      AND (recommendation_status IS NOT NULL OR rating_band IS NOT NULL)""",
 "train_pseudo_unlabeled": """
    SELECT id, body, body_norm FROM rows_split
    WHERE split = 'train' AND id IN (SELECT id FROM masked)""",
 "natural_unlabeled": """
    SELECT id, body, body_norm FROM rows_split
    WHERE split = 'train' AND recommendation_status IS NULL AND rating_band IS NULL""",
 "validation": """
    SELECT id, body, body_norm, recommendation_status, rating_band, rate FROM rows_split
    WHERE split = 'validation'""",
 "test": """
    SELECT id, body, body_norm, recommendation_status, rating_band, rate FROM rows_split
    WHERE split = 'test'""",
}
row_counts = {}
for name, query in FILES.items():
    path = (OUT / f"{name}.csv").as_posix()
    con.execute(f"COPY ({query}) TO '{path}' (HEADER, DELIMITER ',')")
    row_counts[name] = con.execute(
        f"SELECT COUNT(*) FROM read_csv('{path}', all_varchar=true)").fetchall()[0][0]
print(row_counts)

{'train_supervised': 3552000, 'train_pseudo_unlabeled': 736379, 'natural_unlabeled': 18502, 'validation': 922896, 'test': 922896}


In [28]:
import json

total_clean = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{CLEAN.as_posix()}')").fetchall()[0][0]
print("sum of files == clean rows:", sum(row_counts.values()) == total_clean,
      sum(row_counts.values()), total_clean)

def ids_overlap(a, b):
    pa = (OUT / f"{a}.csv").as_posix()
    pb = (OUT / f"{b}.csv").as_posix()
    return con.execute(f"""SELECT COUNT(*) FROM read_csv('{pa}', all_varchar=true) x
                           JOIN read_csv('{pb}', all_varchar=true) y USING (id)""").fetchall()[0][0]

pairs = [("train_supervised","train_pseudo_unlabeled"), ("train_supervised","natural_unlabeled"),
         ("train_pseudo_unlabeled","natural_unlabeled"), ("train_supervised","validation"),
         ("train_supervised","test"), ("validation","test"),
         ("train_pseudo_unlabeled","validation"), ("train_pseudo_unlabeled","test")]
for a, b in pairs:
    print(a, "&", b, "shared ids:", ids_overlap(a, b))

prod_counts = dict(con.execute(
    f"SELECT split, COUNT(*) FROM read_parquet('{PRODUCT_SPLIT.as_posix()}') GROUP BY split").fetchall())

manifest = {
  "split_seed": SPLIT_SEED, "mask_seed": MASK_SEED, "mask_rate": MASK_RATE,
  "split_strategy": "product-grouped; products ordered by md5(product_id|seed); cumulative review count cut at 70% and 85%",
  "rating_bands": {"scale": "1..5 (0 = missing rating)", "low": "1 <= rate < 2.5",
                   "medium": "2.5 <= rate < 4.0", "high": "4.0 <= rate <= 5.0",
                   "encoding": {"low": 0, "medium": 1, "high": 2}},
  "masking": {"hash": "sha256(MASK_SEED|id)[:16] as hex integer",
              "sort": "hash ascending, id ascending as tie-break",
              "rounding": "floor(MASK_RATE * class_count)",
              "labeled_train_rows_per_class": class_counts, "masked_per_class": masked_counts},
  "products_per_split": prod_counts, "rows_per_file": row_counts, "clean_rows_total": total_clean,
}
with open(OUT / "split_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

sum of files == clean rows: True 6152673 6152673
train_supervised & train_pseudo_unlabeled shared ids: 0
train_supervised & natural_unlabeled shared ids: 0
train_pseudo_unlabeled & natural_unlabeled shared ids: 0
train_supervised & validation shared ids: 0
train_supervised & test shared ids: 0
validation & test shared ids: 0
train_pseudo_unlabeled & validation shared ids: 0
train_pseudo_unlabeled & test shared ids: 0
